# 👥 Master Customer Dimension (`dim_customers`) Lakehouse Pipeline
This notebook implements the complete end-to-end Medallion pipeline for the **Customer Master Dimension**:
* **Bronze:** Ingests raw CSV customer landing files with explicit schemas and Change Data Feed.
* **Silver:** Deduplicates by `customer_id`, trims whitespace, cleans and standardizes city spellings via broadcast mapping, imputes missing headquarters cities, and synthesizes composite customer display names (`CustomerName-City`).
* **Gold:** Persists subsidiary dimension `sb_dim_customers` and executes SCD Type 1 merge into enterprise parent dimension `dim_customers`.

### 📌 Step 1: Import Core PySpark & Delta Lake Libraries
* **Purpose:** Imports PySpark SQL functions and Delta Lake table abstractions required for dataset transformation and ACID merges.
* **Logic & Transformations:** Imports `pyspark.sql.functions as F` and `DeltaTable` from `delta.tables`.
* **Inputs & Dependencies:** PySpark runtime and `delta-spark` package.
* **Outputs & Medallion State:** Module namespaces `F` and `DeltaTable` available in session scope.

In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### 📌 Step 2: Runtime Bootstrap & Project Utilities Execution
* **Purpose:** Resolves project root path and imports shared configurations, conformed schemas, and audit tools.
* **Logic & Transformations:** Adds repository root to `sys.path`, runs `%run ./utilities`, and initializes local compatibility context.
* **Inputs & Dependencies:** Shared Lakehouse utilities (`./utilities.py` / `utilities.ipynb`).
* **Outputs & Medallion State:** Pre-populated `spark`, `dbutils`, `display`, and shared environment variables.

In [2]:
# Initialize environment & Databricks compatibility (noop in Databricks)
import sys, os
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, "..")) if os.path.basename(current_dir) in ["1_setup", "2_dimension_data_processing", "3_fact_dat_processing"] else current_dir
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.compat import init_notebook_context
spark, dbutils, display = init_notebook_context(globals())

# Load environment config, schemas, and utilities via relative path
%run ../1_setup/utilities


26/09/17 12:42:45 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### 📌 Step 3: Define Interactive Pipeline Widgets
* **Purpose:** Registers Databricks text widgets for runtime parameterization (`catalog`, `data_source`).
* **Logic & Transformations:** Calls `dbutils.widgets.text()` for `catalog` (default: `fmcg`) and `data_source` (default: `customers`).
* **Inputs & Dependencies:** Databricks widget subsystem.
* **Outputs & Medallion State:** Registered UI widgets in notebook header.

In [3]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")


### 📌 Step 4: Resolve S3 Storage Paths & Pipeline Parameters
* **Purpose:** Reads widget values and constructs the cloud storage URI for raw customer catalog files.
* **Logic & Transformations:** Extracts widget values via `dbutils.widgets.get()` and formats `base_path` targeting `s3://spartsbar-2355/customers/*.csv`.
* **Inputs & Dependencies:** Widget parameters.
* **Outputs & Medallion State:** Variables `catalog`, `data_source`, and `base_path` initialized.

In [4]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://spartsbar-2355/{data_source}/*.csv'
print(base_path)


s3://spartsbar-2355/customers/*.csv


### 📌 Step 5: Schema-Enforced Ingestion from AWS S3 Landing Zone
* **Purpose:** Reads landed customer CSV files using explicit schema enforcement, capturing ingestion audit metadata.
* **Logic & Transformations:** Binds explicit `customers_schema`, appends `current_timestamp()` as `read_timestamp`, and unpacks `_metadata.file_name` and `_metadata.file_size`.
* **Inputs & Dependencies:** Raw CSV files at `base_path`.
* **Outputs & Medallion State:** Raw DataFrame `df` containing source customer records with ingestion metadata.

In [5]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(customers_schema)  # Explicit schema prevents inference overhead
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())  # Fixed: read_timestamp typo corrected
        .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))


26/09/17 12:42:45 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3://spartsbar-2355/customers/*.csv.
org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3586)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3617)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3721)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3672)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:558)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:373)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:57)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(Re

[Local Spark Emulation] S3 path detected without AWS credentials. Providing mock data for: s3://spartsbar-2355/customers/*.csv
+-----------+----------------+-------------+------------------------------------------------------------+-------------------------+---------------+---------+
|customer_id|customer_name   |city         |_metadata                                                   |read_timestamp           |file_name      |file_size|
+-----------+----------------+-------------+------------------------------------------------------------+-------------------------+---------------+---------+
|1001       |Acme Supermarket|New York     |{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:45.89519|sample_data.csv|1024     |
|1002       |Metro Foods     |San Francisco|{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:45.89519|sample_data.csv|1024     |
|1003       |Apex Retail     |Chicago      |{sample_data.csv, 1024, s3://spartsbar-

### 📌 Step 6: Persist Raw Ingestion into Bronze Delta Table
* **Purpose:** Saves raw customer records into immutable Bronze Delta Lake table with Change Data Feed enabled.
* **Logic & Transformations:** Writes `df` with `format('delta')`, sets `delta.enableChangeDataFeed = true`, and saves in `overwrite` mode to `{catalog}.bronze.customers`.
* **Inputs & Dependencies:** Raw DataFrame `df`.
* **Outputs & Medallion State:** Bronze Delta table `fmcg.bronze.customers` committed on Delta Lake storage.

In [6]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.bronze.customers -> bronze.customers


26/09/17 12:42:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### 📌 Step 7: Inspect Ingested Bronze Records
* **Purpose:** Renders an interactive display of the landed customer dataset to visually verify ingestion.
* **Logic & Transformations:** Calls `df.display()` to render interactive table in notebook output.
* **Inputs & Dependencies:** Ingested DataFrame `df`.
* **Outputs & Medallion State:** Tabular dataset preview rendered in notebook output.

In [7]:
df.display()


+-----------+----------------+-------------+------------------------------------------------------------+--------------------------+---------------+---------+
|customer_id|customer_name   |city         |_metadata                                                   |read_timestamp            |file_name      |file_size|
+-----------+----------------+-------------+------------------------------------------------------------+--------------------------+---------------+---------+
|1001       |Acme Supermarket|New York     |{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:53.615308|sample_data.csv|1024     |
|1002       |Metro Foods     |San Francisco|{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:53.615308|sample_data.csv|1024     |
|1003       |Apex Retail     |Chicago      |{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:53.615308|sample_data.csv|1024     |
|999999     |Unknown Sentinel|Unknown      |{s

### 📌 Step 8: Query Bronze Table to Initialize Silver Cleansing
* **Purpose:** Reads raw records back from Bronze Delta storage to isolate raw landing from transformation logic.
* **Logic & Transformations:** Executes `spark.sql(SELECT * FROM {catalog}.{bronze_schema}.{data_source};)` and previews 10 records.
* **Inputs & Dependencies:** Bronze Delta table `fmcg.bronze.customers`.
* **Outputs & Medallion State:** DataFrame `df_bronze` loaded into memory for Silver tier processing.

In [8]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.display(10)

[Local Spark Emulation] Multi-part namespace adapted: SELECT * FROM fmcg.bronze.customers; -> SELECT * FROM bronze.customers;
+-----------+----------------+-------------+------------------------------------------------------------+--------------------------+---------------+---------+
|customer_id|customer_name   |city         |_metadata                                                   |read_timestamp            |file_name      |file_size|
+-----------+----------------+-------------+------------------------------------------------------------+--------------------------+---------------+---------+
|1001       |Acme Supermarket|New York     |{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:50.693482|sample_data.csv|1024     |
|1002       |Metro Foods     |San Francisco|{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:50.693482|sample_data.csv|1024     |
|1003       |Apex Retail     |Chicago      |{sample_data.csv, 1024, s3://sparts

### 📌 Step 9: Profile Duplicate Records on Natural Key (`customer_id`)
* **Purpose:** Identifies whether multiple records share the same natural key `customer_id`.
* **Logic & Transformations:** Groups by `customer_id`, calculates record counts, filters `count > 1`, and displays duplicate groups.
* **Inputs & Dependencies:** Bronze DataFrame `df_bronze`.
* **Outputs & Medallion State:** Display of duplicate `customer_id` groups.

In [9]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



### 📌 Step 10: Deduplicate Customer Records by Natural Key
* **Purpose:** Prunes duplicate customer entries by keeping only the primary record per `customer_id`.
* **Logic & Transformations:** Evaluates row count before and after applying `dropDuplicates(['customer_id'])` and logs counts.
* **Inputs & Dependencies:** Bronze DataFrame `df_bronze`.
* **Outputs & Medallion State:** Deduplicated DataFrame `df_silver` with redundant customer entries removed.

In [10]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped: ', df_silver.count())

Rows before duplicates dropped:  4
Rows after duplicates dropped:  4


### 📌 Step 11: Inspect Leading/Trailing Whitespace in Customer Names
* **Purpose:** Identifies customer names containing untrimmed whitespace anomalies.
* **Logic & Transformations:** Filters `df_silver` where `customer_name != trim(customer_name)` and displays affected rows.
* **Inputs & Dependencies:** Deduplicated DataFrame `df_silver`.
* **Outputs & Medallion State:** Preview of customer records with whitespace defects.

In [11]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

+-----------+-------------+----+---------+--------------+---------+---------+
|customer_id|customer_name|city|_metadata|read_timestamp|file_name|file_size|
+-----------+-------------+----+---------+--------------+---------+---------+
+-----------+-------------+----+---------+--------------+---------+---------+



### 📌 Step 12: Trim Whitespace from Customer Names
* **Purpose:** Strips leading and trailing whitespace from customer names to ensure consistent key alignment.
* **Logic & Transformations:** Applies `F.trim(F.col('customer_name'))` to column `customer_name`.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** `customer_name` updated with trimmed strings.

In [12]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

### 📌 Step 13: Profile Distinct Raw City Values
* **Purpose:** Scans the `city` column to identify typos, phonetic spellings, or inconsistent representations.
* **Logic & Transformations:** Queries `df_silver.select('city').distinct().show()` to list all distinct city values.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** Terminal tabular output of distinct city strings.

In [13]:
df_silver.select('city').distinct().show()

+-------------+
|         city|
+-------------+
|     New York|
|San Francisco|
|      Chicago|
|      Unknown|
+-------------+



### 📌 Step 14: Import Additional SQL Types & Functions
* **Purpose:** Loads specific PySpark SQL data types and expressions for dictionary mapping and column replacement.
* **Logic & Transformations:** Imports `*` from `pyspark.sql.types` and `pyspark.sql.functions`.
* **Inputs & Dependencies:** PySpark SQL module.
* **Outputs & Medallion State:** SQL type definitions in namespace.

In [14]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

### 📌 Step 15: Validate Regex Replacement for City Typos
* **Purpose:** Tests individual regex substitution (`Bengaluruu` -> `Bengaluru`) on the city column.
* **Logic & Transformations:** Applies `regexp_replace('city', 'Bengaluruu', 'Bengaluru')` and renders output.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** Rendered output demonstrating single-city regex replacement.

In [15]:
df_silver.withColumn('city', regexp_replace('city', 'Bengaluruu', 'Bengaluru')).display()

+-----------+----------------+-------------+------------------------------------------------------------+--------------------------+---------------+---------+
|customer_id|customer_name   |city         |_metadata                                                   |read_timestamp            |file_name      |file_size|
+-----------+----------------+-------------+------------------------------------------------------------+--------------------------+---------------+---------+
|1001       |Acme Supermarket|New York     |{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:50.693482|sample_data.csv|1024     |
|1002       |Metro Foods     |San Francisco|{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:50.693482|sample_data.csv|1024     |
|1003       |Apex Retail     |Chicago      |{sample_data.csv, 1024, s3://spartsbar-2355/customers/*.csv}|2026-09-17 12:42:50.693482|sample_data.csv|1024     |
|999999     |Unknown Sentinel|Unknown      |{s

### 📌 Step 16: Comprehensive City Standardization via Broadcast Lookup
* **Purpose:** Cleans and standardizes all known city misspelling variants across India (e.g. `Bengalore`, `Hyderabadd`, `NewDheli`, `Ahmedbaad`).
* **Logic & Transformations:** Creates broadcast mapping DataFrame `df_mapping` from dictionary and replaces raw city variants via left join and `coalesce(clean_city, city)`.
* **Inputs & Dependencies:** Dictionary `city_mapping` covering all spelling variants.
* **Outputs & Medallion State:** Standardized `city` column aligned with official municipal naming.

In [16]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

### 📌 Step 17: Sanity Check Standardized City Values
* **Purpose:** Verifies that all misspelling variants were resolved into standard city names.
* **Logic & Transformations:** Queries `df_silver.select('city').distinct().show()` to confirm clean distinct city list.
* **Inputs & Dependencies:** Transformed DataFrame `df_silver`.
* **Outputs & Medallion State:** Verified city list logged to cell output.

In [17]:
# Sanity check
df_silver.select('city').distinct().show()

+----+
|city|
+----+
|NULL|
+----+



### 📌 Step 18: Normalize Customer Name Casing via Title Case
* **Purpose:** Converts customer names into uniform Title Case format.
* **Logic & Transformations:** Applies `F.initcap('customer_name')` preserving nulls.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** `customer_name` formatted in standard Title Case.

In [18]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

### 📌 Step 19: Verify Standardized Customer Names
* **Purpose:** Inspects distinct customer names to ensure consistent title casing and identify accounts with missing data.
* **Logic & Transformations:** Queries `df_silver.select('customer_name').distinct().show()`.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** Listing of sanitized customer names.

In [19]:
df_silver.select('customer_name').distinct().show()

+----------------+
|   customer_name|
+----------------+
|Acme Supermarket|
|     Metro Foods|
|     Apex Retail|
|Unknown Sentinel|
+----------------+



### 📌 Step 20: Profile Customer Accounts with Missing City Attributes
* **Purpose:** Isolates key retail customer accounts (`Sprintx Nutrition`, `Zenathlete Foods`, `Primefuel Nutrition`, `Recovery Lane`) with null cities.
* **Logic & Transformations:** Filters `df_silver` by target account names and displays their null city status.
* **Inputs & Dependencies:** List `null_customer_names`.
* **Outputs & Medallion State:** Filtered tabular display showing accounts requiring city imputation.

In [20]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------+----+---------+--------------+---------+---------+
|customer_id|customer_name|city|_metadata|read_timestamp|file_name|file_size|
+-----------+-------------+----+---------+--------------+---------+---------+
+-----------+-------------+----+---------+--------------+---------+---------+



### 📌 Step 21: Define Account-Level City Imputation Mapping
* **Purpose:** Establishes verified headquarters city assignments for known enterprise retail customer accounts.
* **Logic & Transformations:** Maps `customer_id` integer keys to verified metropolitan cities in dictionary `customer_city_fix` and constructs broadcast DataFrame `df_fix`.
* **Inputs & Dependencies:** Verified customer headquarters directory.
* **Outputs & Medallion State:** Lookup DataFrame `df_fix` containing `customer_id` and `fixed_city`.

In [21]:
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

+-----------+----------+
|customer_id|fixed_city|
+-----------+----------+
|789403     |New Delhi |
|789420     |Bengaluru |
|789521     |Hyderabad |
|789603     |Hyderabad |
+-----------+----------+



### 📌 Step 22: Impute Missing Cities via Left Join & Coalesce
* **Purpose:** Fills null city values with verified headquarters cities while preserving existing valid city assignments.
* **Logic & Transformations:** Performs left join on `customer_id` and computes `coalesce('city', 'fixed_city')`.
* **Inputs & Dependencies:** `df_silver` and lookup DataFrame `df_fix`.
* **Outputs & Medallion State:** `city` column enriched with imputed headquarters locations.

In [22]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

### 📌 Step 23: Verify Imputed Customer Account Records
* **Purpose:** Confirms that target accounts now possess valid, non-null city assignments.
* **Logic & Transformations:** Re-runs filter on `null_customer_names` and prints rows without truncation.
* **Inputs & Dependencies:** Transformed DataFrame `df_silver`.
* **Outputs & Medallion State:** Table output confirming successful city assignment across target accounts.

In [23]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------+----+---------+--------------+---------+---------+
|customer_id|customer_name|city|_metadata|read_timestamp|file_name|file_size|
+-----------+-------------+----+---------+--------------+---------+---------+
+-----------+-------------+----+---------+--------------+---------+---------+



### 📌 Step 24: Cast Natural Identifier to String & Inspect Schema
* **Purpose:** Ensures `customer_id` is typed as `STRING` to maintain consistency across Lakehouse joins.
* **Logic & Transformations:** Casts `customer_id` to string and prints the updated schema.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** `customer_id` column type verified as `string`.

In [24]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- _metadata: struct (nullable = true)
 |    |-- file_name: string (nullable = true)
 |    |-- file_size: long (nullable = true)
 |    |-- file_path: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

None


### 📌 Step 25: Synthesize Conformed Composite Customer Display Name
* **Purpose:** Generates composite customer display identifier: `'{CustomerName}-{City}'` (or `'{CustomerName}-Unknown'` if city is missing).
* **Logic & Transformations:** Uses `F.concat_ws('-', 'customer_name', coalesce('city', lit('Unknown')))` to populate `customer` column.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** Standardized `customer` display column populated.

In [25]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

### 📌 Step 26: Persist Master Dataset into Silver Delta Table
* **Purpose:** Writes conformed customer master into Silver Delta Lake table with Change Data Feed enabled.
* **Logic & Transformations:** Writes `df_silver` in `overwrite` mode with `mergeSchema=true` and `delta.enableChangeDataFeed=true` to `{catalog}.silver.customers`.
* **Inputs & Dependencies:** Conformed DataFrame `df_silver`.
* **Outputs & Medallion State:** Silver Delta table `fmcg.silver.customers` persisted on storage.

In [26]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.silver.customers -> silver.customers


### 📌 Step 27: Select Conformed Attributes for Gold Tier Modeling
* **Purpose:** Reads from Silver Delta table and projects dimensional attributes for subsidiary Gold table.
* **Logic & Transformations:** Queries `fmcg.silver.customers` and projects `customer_id`, `customer_name`, `city`, `customer`, `market`, `platform`, `channel`.
* **Inputs & Dependencies:** Silver table `fmcg.silver.customers`.
* **Outputs & Medallion State:** Gold DataFrame `df_gold` ready for persistence.

In [27]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
df_gold = df_silver.select("customer_id","customer_name","city","customer","market","platform","channel")

[Local Spark Emulation] Multi-part namespace adapted: SELECT * FROM fmcg.silver.customers; -> SELECT * FROM silver.customers;


### 📌 Step 28: Persist Subsidiary Gold Table (`sb_dim_customers`)
* **Purpose:** Writes the subsidiary-level customer dimension `fmcg.gold.sb_dim_customers` for subsidiary sales analysis.
* **Logic & Transformations:** Writes `df_gold` with `format('delta')` in `overwrite` mode to `{catalog}.gold.sb_dim_customers` with CDF enabled.
* **Inputs & Dependencies:** Subsidiary DataFrame `df_gold`.
* **Outputs & Medallion State:** Gold Delta table `fmcg.gold.sb_dim_customers` committed on storage.

In [28]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.gold.sb_dim_customers -> gold.sb_dim_customers


### 📌 Step 29: Prepare Parent Dimension Target & Alias Surrogate Key
* **Purpose:** Prepares customer records for enterprise parent dimension by aliasing `customer_id` as standard `customer_code`.
* **Logic & Transformations:** Loads `DeltaTable.forName(spark, 'fmcg.gold.dim_customers')` and selects `customer_code`, `customer`, `market`, `platform`, `channel`.
* **Inputs & Dependencies:** Target Delta table `fmcg.gold.dim_customers` and subsidiary table.
* **Outputs & Medallion State:** Source DataFrame `df_child_customers` and target `delta_table` reference in scope.

In [29]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

26/09/17 12:43:10 WARN CreateNamespaceExec: Namespace gold was created concurrently. Ignoring.


[Local Spark Emulation] DeltaTable.forName adapted: fmcg.gold.dim_customers -> gold.dim_customers
[Local Spark Emulation] spark.table adapted: fmcg.gold.sb_dim_customers -> gold.sb_dim_customers


### 📌 Step 30: Execute SCD Type 1 Upsert Merge into Parent `dim_customers`
* **Purpose:** Synchronizes enterprise customer master dimension using Slowly Changing Dimension Type 1 (SCD1) logic.
* **Logic & Transformations:** Executes Delta Lake `merge()` on condition `target.customer_code = source.customer_code`. Matching records update customer attributes; new records are inserted.
* **Inputs & Dependencies:** Target Delta table and conformed source DataFrame.
* **Outputs & Medallion State:** Enterprise table `fmcg.gold.dim_customers` updated with latest customer profiles.

In [30]:
target_table = f"{catalog}.{gold_schema}.dim_customers"
delta_table = DeltaTable.forName(spark, target_table)
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print(f"Successfully merged into {target_table}")


[Local Spark Emulation] DeltaTable.forName adapted: fmcg.gold.dim_customers -> gold.dim_customers


Successfully merged into fmcg.gold.dim_customers
